# Prompt 01: Why Prompt Engineering Still Matters

Hands-on demo that motivates the rest of the module.

1. Same intent, different prompts — observe accuracy delta
2. Same intent, different wordings — observe cost delta
3. Cross-provider prompt portability check
4. Exercises: prompt-cost estimator, reproduce a known prompt-sensitivity failure

Deps: `pip install anthropic openai tiktoken`

## 1. Shared Caller

In [ ]:
import os, time

def call(system, user, max_tokens=300, temperature=0):
    t0 = time.perf_counter()
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(
            model='claude-sonnet-4-6', max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{'role':'user','content':user}])
        return dict(text=r.content[0].text,
                    in_tok=r.usage.input_tokens, out_tok=r.usage.output_tokens,
                    ms=(time.perf_counter()-t0)*1000)
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model='gpt-4o-mini', max_tokens=max_tokens, temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return dict(text=r.choices[0].message.content,
                    in_tok=r.usage.prompt_tokens, out_tok=r.usage.completion_tokens,
                    ms=(time.perf_counter()-t0)*1000)
    return dict(text='[no provider]', in_tok=0, out_tok=0, ms=0)

## 2. Same Intent, Different Prompts — Accuracy Delta

Task: extract the year from a sentence. Watch how different phrasings change output shape.

In [ ]:
input_text = 'The Treaty of Versailles was signed on June 28, 1919, ending World War I.'

variants = [
    ('vague',           'What year?'),
    ('specific',        'Return only the 4-digit year mentioned. No other text.'),
    ('xml + specific',  '<task>Return only the 4-digit year mentioned.</task>\n<text>' + input_text + '</text>'),
    ('few-shot',        'Return the year only.\n\nInput: Born 1820. Year: 1820\nInput: In 1776 we declared... Year: 1776\nInput: ' + input_text + ' Year:'),
]

for name, prompt in variants:
    system = 'You are a precise extraction system.'
    user = prompt if '<task>' in prompt or 'Input:' in prompt else f'{prompt}\n\n{input_text}'
    r = call(system, user, max_tokens=50)
    print(f'[{name:<18}] out_tok={r["out_tok"]:>3}   text={r["text"].strip()[:80]!r}')

Even on a model as capable as Sonnet, the 'vague' variant often includes explanatory prose; 'specific' and 'few-shot' usually don't. That output-length gap is a direct 2–5x output-token cost delta.

## 3. Cost Delta on 100 Repeats

In [ ]:
# Run each variant N times to get stable average output tokens, then compute daily cost.

N = 5
daily_calls = 100_000
price_in_per_M = 3.0      # claude sonnet input
price_out_per_M = 15.0    # claude sonnet output

print(f"{'variant':<18} {'avg in_tok':>10} {'avg out_tok':>12} {'daily $':>9}")
for name, prompt in variants:
    user = prompt if '<task>' in prompt or 'Input:' in prompt else f'{prompt}\n\n{input_text}'
    in_tot = out_tot = 0
    for _ in range(N):
        r = call('You are a precise extraction system.', user, max_tokens=50)
        in_tot += r['in_tok']; out_tot += r['out_tok']
    avg_in, avg_out = in_tot/N, out_tot/N
    daily = ((avg_in * price_in_per_M) + (avg_out * price_out_per_M)) / 1_000_000 * daily_calls
    print(f'{name:<18} {avg_in:>10.1f} {avg_out:>12.1f} ${daily:>8.2f}')

A "vague" vs "specific" prompt difference at 100K calls/day can easily be ~$500/day in output tokens alone. This is why prompts are a cost lever, not just a quality lever.

## 4. Cross-Provider Portability

A prompt that works on Claude sometimes fails on GPT and vice versa. Build your evals to cover both.

In [ ]:
# Run the same 'xml + specific' variant against both providers if keys available.
xml_prompt = '<task>Return only the 4-digit year mentioned.</task>\n<text>' + input_text + '</text>'
sys = 'You are a precise extraction system.'

providers = []
if os.getenv('ANTHROPIC_API_KEY'):
    providers.append(('claude', {'ANTHROPIC_API_KEY': os.environ['ANTHROPIC_API_KEY']}))
if os.getenv('OPENAI_API_KEY'):
    providers.append(('gpt-4o-mini', {'OPENAI_API_KEY': os.environ['OPENAI_API_KEY']}))

if len(providers) < 2:
    print('Set both ANTHROPIC_API_KEY and OPENAI_API_KEY to compare.')
else:
    for name, env in providers:
        orig = dict(os.environ)
        os.environ.clear(); os.environ.update(env)
        r = call(sys, xml_prompt)
        print(f'[{name:<12}] {r["text"].strip()!r}')
        os.environ.clear(); os.environ.update(orig)

## 5. Exercise: Cost Estimator for One Feature

Pick one feature you're shipping (or imagine one). Write a function that, given a candidate prompt and an expected traffic pattern:
- Estimates avg input/output tokens on 20 sample inputs.
- Returns projected daily and monthly cost.

Use this to sanity-check prompt changes before they land.

## 6. Exercise: Reproduce a Known Sensitivity Bug

Try these two prompts against the same model and same question:

- `"Is 9.11 greater than 9.8?"`
- `"Which is greater: 9.11 or 9.8? Respond with just the number."`

The first often trips classic LLM numeric bugs; the second usually fixes it. The exercise: find three more such pairs in your own domain. These are the regression cases worth locking into your eval set.

## 7. Takeaways

- **Prompts are the product surface** the model sees — design them.
- **Wording changes cost and quality**, often dramatically.
- **Cross-provider portability** takes deliberate effort.
- **Iterate with measurement**; the eval loop is the only reliable compass.
- **Treat prompts like code** — version, review, test, roll back.

Next: **02 — Anatomy of a Prompt** — decompose the parts and see what each contributes.